# ALTE Common Corpus SIG — Drive-aware pipeline runner

This notebook is aligned with the existing June 2026 data in Google Drive and the current sense-aware scripts in GitHub.

## Data-preservation rules

- The existing language folders (`en`, `fr`, `es`, `de`, `cs`) are treated as a **frozen legacy function-only run**.
- Nothing in those folders is deleted, renamed or overwritten.
- Reusable legacy inputs are read directly.
- All new sense-aware outputs are written under `sense_aware_v2/`.
- Write cells are disabled by default. Set `ALLOW_NEW_WRITES = True` only when you intend to create or resume v2 outputs.
- Existing v2 files are resumed by row ID where supported; the notebook does not remove them.


## 1. Mount Drive and obtain the current repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path('/content/ALTE-Common-Corpus-SIG')
DRIVE_ROOT = Path('/content/drive/MyDrive/ALTE-Common-Corpus-SIG')

if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/Pertam/ALTE-Common-Corpus-SIG.git',
        str(REPO_ROOT)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_stage1_5.txt'], check=False)

print('Repository:', REPO_ROOT)
print('Drive data:', DRIVE_ROOT)


## 2. Define the legacy and v2 layouts

In [ ]:
LANGUAGES = ['en', 'fr', 'es', 'de', 'cs']
LANGUAGE_NAMES = {
    'en': 'English', 'fr': 'French', 'es': 'Spanish',
    'de': 'German', 'cs': 'Czech',
}

V2_ROOT = DRIVE_ROOT / 'sense_aware_v2'
DRIVE_TAXONOMY = DRIVE_ROOT / 'taxonomy' / 'cefr_function_taxonomy_v0_2.csv'
REPO_TAXONOMY = REPO_ROOT / 'taxonomy' / 'cefr_function_taxonomy_v0_2.csv'
TAXONOMY = DRIVE_TAXONOMY if DRIVE_TAXONOMY.exists() else REPO_TAXONOMY

ALLOW_NEW_WRITES = False


def legacy_paths(lang):
    base = DRIVE_ROOT / lang
    return {
        'raw': base / 'stage00_raw_sentences' / f'{lang}_sentences.txt',
        'prepared': base / 'stage01_prepared_sentences' / f'stage01_{lang}_sentences.parquet',
        'lemma_index': base / 'stage02_tokenise_lemmatise' / f'{lang}_lemma_sentence_index.parquet',
        'token_parts': base / 'stage02_tokenise_lemmatise' / 'token_parts',
        'stats': base / 'stage03_lemma_stats' / f'stage03_{lang}_lemma_frequencies.csv',
        'full_sample': base / 'stage04_samples' / f'stage04_{lang}_random_15_lemmas_all_sentences.csv',
        'test50': base / 'stage04_samples' / f'stage04_{lang}_dispersed_test50_normalised_for_stage05.csv',
        'function_pass1': base / 'stage05_llm_tagging' / f'stage05_{lang}_pass1_dispersed_test50.csv',
        'legacy_function_pass2': base / 'stage05_llm_tagging' / f'stage05_{lang}_pass2_dispersed_test50.csv',
        'legacy_function_pass3': base / 'stage05_llm_tagging' / f'stage05_{lang}_pass3_dispersed_test50.csv',
        'legacy_final': base / 'stage06_final_dataset' / f'stage06_{lang}_final_dispersed_test50.csv',
    }


def v2_paths(lang):
    base = V2_ROOT / lang
    return {
        'input': base / 'inputs' / f'{lang}_test50_target_occurrences.csv',
        'inventory': base / 'sense_inventory' / f'{lang}_sense_inventory_v1.csv',
        'sense_pass1': base / 'sense_pass1' / f'{lang}_sense_pass1_test50.csv',
        'sense_pass2': base / 'sense_pass2' / f'{lang}_sense_pass2_informed_test50.csv',
        'function_pass2': base / 'function_pass2' / f'{lang}_function_pass2_informed_test50.csv',
        'sense_pass3': base / 'adjudication' / f'{lang}_sense_pass3_problem_cases.csv',
        'function_pass3': base / 'adjudication' / f'{lang}_function_pass3_problem_cases.csv',
        'combined': base / 'combined_review' / f'{lang}_combined_sense_function_test50.csv',
        'blind_sample': base / 'blind_validation' / f'{lang}_blind_sample.csv',
        'sense_pass2_blind': base / 'blind_validation' / f'{lang}_sense_pass2_blind.csv',
        'function_pass2_blind': base / 'blind_validation' / f'{lang}_function_pass2_blind.csv',
    }

print('Taxonomy selected:', TAXONOMY)
print('New outputs will use:', V2_ROOT)
print('ALLOW_NEW_WRITES =', ALLOW_NEW_WRITES)


## 3. Run the read-only alignment audit

This checks:

- the existing Stage 0–6 files for all five languages;
- required columns and 50-row test-chain joins;
- whether the June Pass 1 files can be reused;
- whether old Pass 2, Pass 3 and Stage 6 outputs are legacy reference files rather than current sense-aware outputs;
- taxonomy alignment;
- any v2 files that already exist.

The audit does not change project data. It creates only a new timestamped report in `audit_reports/`.


In [ ]:
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
audit_dir = DRIVE_ROOT / 'audit_reports'
audit_csv = audit_dir / f'sense_aware_alignment_audit_{stamp}.csv'
audit_json = audit_dir / f'sense_aware_alignment_audit_{stamp}.json'

subprocess.run([
    sys.executable,
    str(REPO_ROOT / 'scripts' / '00_audit_drive_outputs.py'),
    '--drive_root', str(DRIVE_ROOT),
    '--repo_root', str(REPO_ROOT),
    '--report_csv', str(audit_csv),
    '--report_json', str(audit_json),
], check=True)


## 4. Create v2-compatible input copies

The existing normalised 50-row samples are not edited. This step creates small copies under `sense_aware_v2/` and adds explicit target/provenance columns required by the revised workflow.

`target_token` remains blank because the June sample records the target lemma and POS but not the exact surface token. The lemma, POS and sentence remain sufficient for the pilot sense-review step, but this limitation must be retained in provenance.


In [ ]:
import pandas as pd


def require_writes():
    if not ALLOW_NEW_WRITES:
        raise RuntimeError('Set ALLOW_NEW_WRITES = True before creating new v2 files.')


def create_v2_inputs():
    require_writes()
    for lang in LANGUAGES:
        source = legacy_paths(lang)['test50']
        output = v2_paths(lang)['input']
        if output.exists():
            print('Keeping existing v2 input:', output)
            continue
        frame = pd.read_csv(source, dtype=str, encoding='utf-8-sig').fillna('')
        required = ['row_id', 'sentence', 'lemma', 'pos']
        missing = [column for column in required if column not in frame.columns]
        if missing:
            raise ValueError(f'{source} is missing {missing}')
        if 'language' not in frame.columns:
            frame['language'] = LANGUAGE_NAMES[lang]
        frame['target_token'] = ''
        frame['target_lemma'] = frame['lemma']
        frame['target_pos'] = frame['pos']
        frame['source_pipeline'] = 'legacy_june_2026_function_only_run'
        frame['source_file'] = str(source)
        frame['target_token_status'] = 'not_recorded_in_legacy_sample'
        output.parent.mkdir(parents=True, exist_ok=True)
        frame.to_csv(output, index=False)
        print(f'Created {output} ({len(frame)} rows)')

# Uncomment after setting ALLOW_NEW_WRITES = True:
# create_v2_inputs()


## 5. Create provisional sense inventories

Use the corrected full-sentence sample as evidence, not only the 50-row function-tagging sample. The script reads at most 50 examples per lemma.

This produces **provisional** inventories. A language expert must revise them and set retained rows to `inventory_status=approved` before production sense tagging.


In [ ]:
def run_command(arguments):
    print('RUN:', ' '.join(map(str, arguments)))
    subprocess.run([str(x) for x in arguments], check=True, cwd=REPO_ROOT)


def create_sense_inventories(limit_lemmas=0, dry_run=False):
    require_writes()
    for lang in LANGUAGES:
        output = v2_paths(lang)['inventory']
        command = [
            sys.executable, 'scripts/04a_create_sense_inventory.py',
            '--samples', legacy_paths(lang)['full_sample'],
            '--output', output,
            '--max_examples', '50',
        ]
        if limit_lemmas:
            command += ['--limit_lemmas', str(limit_lemmas)]
        if dry_run:
            command += ['--dry_run']
        run_command(command)

# Safe first test after enabling writes:
# create_sense_inventories(limit_lemmas=1, dry_run=True)
# Production proposal run:
# create_sense_inventories()


### Mandatory human checkpoint

Open each inventory CSV in Drive. Merge, split, rename or remove candidate senses as needed, retain `OTHER` and `UNCLEAR`, and mark retained rows `approved`. Do not run the production sense passes until this is complete.


## 6. Run Sense Pass 1

In [ ]:
def run_sense_pass1(limit=0, dry_run=False, allow_provisional=False):
    require_writes()
    for lang in LANGUAGES:
        paths = v2_paths(lang)
        command = [
            sys.executable, 'scripts/04b_run_sense_pass1.py',
            '--samples', paths['input'],
            '--inventory', paths['inventory'],
            '--output', paths['sense_pass1'],
        ]
        if limit:
            command += ['--limit', str(limit)]
        if dry_run:
            command += ['--dry_run']
        if allow_provisional:
            command += ['--allow_provisional']
        run_command(command)

# Schema test only, after creating v2 inputs and an inventory:
# run_sense_pass1(limit=5, dry_run=True, allow_provisional=True)
# Production, after human approval:
# run_sense_pass1()


## 7. Reuse the June Function Pass 1

The June Function Pass 1 files have the columns required by the current scripts and were produced independently of lexical-sense labels. They are therefore reused as the initial function proposals.

The June Pass 2, Pass 3 and Stage 6 files are retained for historical comparison only. They are not supplied to the revised sense-aware builder.


In [ ]:
for lang in LANGUAGES:
    path = legacy_paths(lang)['function_pass1']
    print(lang, 'legacy Function Pass 1:', path, 'exists=', path.exists())


## 8. Run the informed Pass 2 reviews

In [ ]:
def run_informed_pass2(limit=0, dry_run=False):
    require_writes()
    for lang in LANGUAGES:
        legacy = legacy_paths(lang)
        paths = v2_paths(lang)

        sense_command = [
            sys.executable, 'scripts/04c_run_sense_pass2.py',
            '--samples', paths['input'],
            '--inventory', paths['inventory'],
            '--pass1', paths['sense_pass1'],
            '--function_pass1', legacy['function_pass1'],
            '--output', paths['sense_pass2'],
        ]
        function_command = [
            sys.executable, 'scripts/05b_run_pass2.py',
            '--sentences', paths['input'],
            '--taxonomy', TAXONOMY,
            '--pass1', legacy['function_pass1'],
            '--sense_pass1', paths['sense_pass1'],
            '--output', paths['function_pass2'],
        ]
        if limit:
            sense_command += ['--limit', str(limit)]
            function_command += ['--limit', str(limit)]
        if dry_run:
            sense_command += ['--dry_run']
            function_command += ['--dry_run']
        run_command(sense_command)
        run_command(function_command)

# Schema test:
# run_informed_pass2(limit=5, dry_run=True)
# Production:
# run_informed_pass2()


## 9. Targeted adjudication

In [ ]:
def run_targeted_adjudication(limit=0):
    require_writes()
    for lang in LANGUAGES:
        legacy = legacy_paths(lang)
        paths = v2_paths(lang)
        sense_command = [
            sys.executable, 'scripts/04d_run_sense_adjudication.py',
            '--pass1', paths['sense_pass1'],
            '--pass2', paths['sense_pass2'],
            '--inventory', paths['inventory'],
            '--only_problem_cases',
            '--output', paths['sense_pass3'],
        ]
        function_command = [
            sys.executable, 'scripts/05c_run_pass3.py',
            '--pass1', legacy['function_pass1'],
            '--pass2', paths['function_pass2'],
            '--taxonomy', TAXONOMY,
            '--only_problem_cases',
            '--output', paths['function_pass3'],
        ]
        if limit:
            sense_command += ['--limit', str(limit)]
            function_command += ['--limit', str(limit)]
        run_command(sense_command)
        run_command(function_command)

# This step uses the API and has no dry-run mode:
# run_targeted_adjudication(limit=5)
# run_targeted_adjudication()


## 10. Build the combined sense-and-function review files

In [ ]:
def build_combined_review_files():
    require_writes()
    for lang in LANGUAGES:
        legacy = legacy_paths(lang)
        paths = v2_paths(lang)
        command = [
            sys.executable, 'scripts/06_make_final_dataset.py',
            '--samples', paths['input'],
            '--sense_pass1', paths['sense_pass1'],
            '--sense_pass2', paths['sense_pass2'],
            '--function_pass1', legacy['function_pass1'],
            '--function_pass2', paths['function_pass2'],
            '--output', paths['combined'],
        ]
        if paths['sense_pass3'].exists():
            command += ['--sense_pass3', paths['sense_pass3']]
        if paths['function_pass3'].exists():
            command += ['--function_pass3', paths['function_pass3']]
        run_command(command)

# build_combined_review_files()


## 11. Optional blind-validation sample

Blind review is not the production default. This cell creates a separate, reproducible 15% sample and stores all blind outputs separately.


In [ ]:
def create_blind_samples(fraction=0.15, seed=20260711):
    require_writes()
    for lang in LANGUAGES:
        source = v2_paths(lang)['input']
        output = v2_paths(lang)['blind_sample']
        if output.exists():
            print('Keeping existing blind sample:', output)
            continue
        frame = pd.read_csv(source, dtype=str).fillna('')
        blind = frame.sample(frac=fraction, random_state=seed).sort_values('row_id')
        output.parent.mkdir(parents=True, exist_ok=True)
        blind.to_csv(output, index=False)
        print(f'Created {output} ({len(blind)} rows)')


def run_blind_pass2(dry_run=False):
    require_writes()
    for lang in LANGUAGES:
        paths = v2_paths(lang)
        sense_command = [
            sys.executable, 'scripts/04c_run_sense_pass2.py',
            '--samples', paths['blind_sample'],
            '--inventory', paths['inventory'],
            '--blind',
            '--output', paths['sense_pass2_blind'],
        ]
        function_command = [
            sys.executable, 'scripts/05b_run_pass2.py',
            '--sentences', paths['blind_sample'],
            '--taxonomy', TAXONOMY,
            '--blind',
            '--output', paths['function_pass2_blind'],
        ]
        if dry_run:
            sense_command += ['--dry_run']
            function_command += ['--dry_run']
        run_command(sense_command)
        run_command(function_command)

# create_blind_samples()
# run_blind_pass2(dry_run=True)
# run_blind_pass2()


## 12. Re-run the audit after any new work

Re-run Section 3 after each major stage. The audit recognises the old June outputs as preserved legacy evidence and checks new files under `sense_aware_v2/` separately.
